# DataObject — Alias-Driven Structured Access to Sensor Data

The `DataObject` class bridges query metadata to time-series data with predictable, alias-based column names and built-in entity grouping.

**Before** (fragile positional indexing):
```python
df = query.latest_data(cast_value='float')
if df[0,1] > 75:  # what is column 1?
    ...
```

**After** (alias-driven):
```python
data = query.data(limit=1, order="desc", cast_value="float")
for basin_uri, group in data.by("basin"):
    cl = group["chlorine"]
    if cl["value"][0] > 75:
        ...
```

This notebook walks through the key features using the test CSV dataset.

## Setup

Connect to a running Acquirium server and load the test graph. Make sure containers are running (`make up` or `make testing-up`).

In [ ]:
import time
from acquirium import Acquirium, DataObject
from acquirium.Client.query import Query
from acquirium.internals.internals_namespaces import ACQUIRIUM_NS

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

# Load the test graph (CSV-backed sensors)
acq.insert_graph("../tests/test_model_csv.ttl")

# Wait for ingestion to complete
time.sleep(1)
status = acq.client.ingest_status()
while status["done"] < status["total"] - status["error"]:
    time.sleep(2)
    status = acq.client.ingest_status()
print(f"Ingestion complete: {status}")

## 1. Basic Usage — `query.data()`

Call `.data()` on any query that has data nodes. This returns a `DataObject` instead of a raw DataFrame.

In [ ]:
# Find all data points in the graph
query = acq.find_all_data()
query.metadata_head()

# Get a DataObject with all timeseries (first 10 rows per stream)
data = query.data(limit=10)
print(data)

## 2. Alias-Based Access — `data["alias"]`

Access time-series by the alias you gave the data node in your query. Returns a clean `[time, value]` DataFrame.

In [ ]:
# See what aliases are available
print("Data aliases:", data.aliases)

# Access by alias name — returns [time, value]
first_alias = data.aliases[0]
df = data[first_alias]
print(f"\ndata['{first_alias}']:")
df

## 3. Named Aliases with Entity + Data Queries

When you build a query with `find_entity` + `find_data`, you get meaningful alias names and entity context for grouping.

In [ ]:
# Build a query: find entities of class B, then their related data
entity_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="equipment")
        .find_data(alias="sensor_data")
)
entity_query.show_query_graph()

# Fetch data with named aliases
data = entity_query.data(limit=5, order="desc", cast_value="float")
print(data)
print("\nData aliases:", data.aliases)
print("Entity aliases:", data.entity_aliases)

In [ ]:
# Access the named alias directly
data["sensor_data"]

## 4. Grouping by Entity — `data.by("alias")`

When your query includes entity nodes, you can group the data by entity. This is useful for "for each X, get its sensors" patterns.

In [ ]:
# Group by the "equipment" entity
for entity_uri, group in data.by("equipment"):
    print(f"\nEquipment: {entity_uri}")
    print(f"  Aliases in group: {group.aliases}")
    sensor_df = group["sensor_data"]
    print(f"  Rows: {len(sensor_df)}")
    print(sensor_df)

## 5. Flat DataFrames — `data.dataframe()`

Get a standard wide or narrow DataFrame when you need one for plotting or further analysis.

In [ ]:
# Wide format: [time, alias_col_1, alias_col_2, ...]
wide_df = data.dataframe(shape="wide")
print("Wide DataFrame:")
wide_df

In [ ]:
# Narrow format: the full enriched tall frame
narrow_df = data.dataframe(shape="narrow")
print("Narrow DataFrame:")
narrow_df

## 6. Iterating Individual Series — `data.iter("alias")`

When you need to process each point's timeseries individually (e.g., per-sensor anomaly detection).

In [ ]:
# Iterate over each individual point's timeseries
for point_uri, series_df in data.iter("sensor_data"):
    print(f"Point: {point_uri}  |  rows: {len(series_df)}  |  mean: {series_df['value'].mean():.2f}")

## 7. Metadata & Introspection

Inspect what's in the DataObject without looking at the raw timeseries.

In [ ]:
# Metadata: unique combinations of alias, point_uri, ref_uri, and entity URIs
print("Metadata:")
data.metadata()

In [ ]:
# Reference info: which external refs back a given alias
print("Ref info for 'sensor_data':")
for idx, ref_uri in data.ref_info("sensor_data"):
    print(f"  [{idx}] {ref_uri}")

In [ ]:
# Latest value for an alias
print("Latest value:")
data.latest("sensor_data")

## 8. Multi-Level Query Example

A more complex query with multiple entity levels and data nodes, showing how grouping and alias access compose together.

In [ ]:
# Multi-level: find B entities, their related E entities, and all their data
multi_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="parent")
       .find_related(_class=ACQUIRIUM_NS.E, alias="child")
       .find_all_data(alias="readings")
)
multi_query.show_query_graph()

multi_data = multi_query.data(limit=3, order="desc", cast_value="float")
print(multi_data)
print("\nEntity aliases:", multi_data.entity_aliases)

In [ ]:
# Group by child entity and summarize
if "child" in multi_data.entity_aliases:
    for child_uri, group in multi_data.by("child"):
        readings = group["readings"]
        if not readings.is_empty():
            print(f"Child: {child_uri}")
            print(f"  Latest value: {readings.sort('time', descending=True)['value'][0]}")
            print(f"  Mean: {readings['value'].mean():.2f}")
            print()

## 9. App Pattern — Threshold Alert (Before vs After)

The `DataObject` API makes app logic much more readable and robust.

In [ ]:
# --- BEFORE: fragile positional indexing ---
# df = query.latest_data(cast_value='float')
# if df[0,1] > 75:   # What is column 1? What if column order changes?
#     print("Alert!")

# --- AFTER: alias-driven with DataObject ---
# Simulating the threshold app pattern with our test data
threshold_query = (
    acq.find_entity(_class=ACQUIRIUM_NS.B, alias="equipment")
       .find_data(alias="sensor")
)

data = threshold_query.data(limit=1, order="desc", cast_value="float")

for equip_uri, group in data.by("equipment"):
    sensor = group["sensor"]
    if sensor.is_empty():
        print(f"{equip_uri}: No data")
    else:
        val = sensor["value"][0]
        ts = sensor["time"][0]
        print(f"{equip_uri}: value={val:.2f} at {ts}")

## API Reference

| Method / Property | Returns | Description |
|---|---|---|
| `query.data(start, end, limit, order, cast_value)` | `DataObject` | Construct from a query |
| `data["alias"]` | `pl.DataFrame` | `[time, value]` for single-ref; `[time, value, ref_uri]` for multi-ref |
| `data.by("entity_alias")` | `Iterator[(str, DataObject)]` | Group by entity, yields `(uri, sub_DataObject)` |
| `data.dataframe(shape="wide")` | `pl.DataFrame` | Pivoted `[time, alias_1, alias_2, ...]` |
| `data.dataframe(shape="narrow")` | `pl.DataFrame` | Full enriched tall frame |
| `data.iter("alias")` | `Iterator[(str, pl.DataFrame)]` | Per-point `(point_uri, [time, value])` |
| `data.metadata()` | `pl.DataFrame` | Unique `(data_alias, point_uri, ref_uri, entity__*)` |
| `data.latest("alias")` | `pl.DataFrame` | Most recent `[time, value]` |
| `data.ref_info("alias")` | `list[(int, str)]` | Indexed ref URIs |
| `data.aliases` | `list[str]` | Available data aliases |
| `data.entity_aliases` | `list[str]` | Available entity aliases |
| `data.is_empty()` | `bool` | Whether any data exists |